In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
np.random.seed(42)
torch.manual_seed(42)

In [3]:
df = pd.read_csv('iris.csv')

df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
list(df.columns)

['Id',
 'SepalLengthCm',
 'SepalWidthCm',
 'PetalLengthCm',
 'PetalWidthCm',
 'Species']

In [5]:
df.shape

(150, 6)

In [6]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

In [7]:
df = df.drop(columns=['Id'])
df.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [8]:
species_to_number = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2
}

encoded_targets = df['Species'].map(species_to_number)
targets_series = encoded_targets

print(encoded_targets.head())
print(encoded_targets.value_counts())

del df['Species']

print()
print(list(df.columns))

0    0
1    0
2    0
3    0
4    0
Name: Species, dtype: int64
Species
0    50
1    50
2    50
Name: count, dtype: int64

['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']


### Train-test split

In [10]:
all_inputs_numpy = df.to_numpy()
all_targets_numpy = targets_series.to_numpy()

number_of_rows = all_inputs_numpy.shape[0]

all_indices = np.arange(number_of_rows)
shuffled_indices = np.random.permutation(all_indices)

number_of_testing_rows = int(number_of_rows * 0.2)
number_of_training_rows = number_of_rows - number_of_testing_rows

training_indices = shuffled_indices[:number_of_training_rows]
testing_indices = shuffled_indices[number_of_training_rows:]

training_inputs_numpy = all_inputs_numpy[training_indices]
testing_inputs_numpy = all_inputs_numpy[testing_indices]
training_targets_numpy = all_targets_numpy[training_indices]
testing_targets_numpy = all_targets_numpy[testing_indices]

In [11]:
print('training input: ', training_inputs_numpy.shape)
print('testing input: ', testing_inputs_numpy.shape)
print('training targets: ', training_targets_numpy.shape)
print('testing target shape: ', testing_targets_numpy.shape)

training input:  (120, 4)
testing input:  (30, 4)
training targets:  (120,)
testing target shape:  (30,)


In [13]:
# + 1e-6 is for the edge case to avoid the null division
training_inputs_normalized = (training_inputs_numpy - training_inputs_numpy.mean(axis=0)) / (training_inputs_numpy.std(axis=0) + 1e-6)
testing_inputs_normalized = (testing_inputs_numpy - training_inputs_numpy.mean(axis=0)) / (training_inputs_numpy.std(axis=0) + 1e-6)

In [14]:
training_inputs_tensor = torch.tensor(training_inputs_normalized, dtype=torch.float32)
testing_inputs_tensor = torch.tensor(testing_inputs_normalized, dtype=torch.float32)

training_targets_tensor = torch.tensor(training_targets_numpy, dtype=torch.long)
testing_targets_tensor = torch.tensor(testing_targets_numpy, dtype=torch.long)

print("train input tensor: ", training_inputs_tensor.shape, training_inputs_tensor.dtype)
print("train target tensor:", training_targets_tensor.shape, training_targets_tensor.dtype)
print("test input tensor:  ", testing_inputs_tensor.shape, testing_inputs_tensor.dtype)
print("test target tensor: ", testing_targets_tensor.shape, testing_targets_tensor.dtype)

train input tensor:  torch.Size([120, 4]) torch.float32
train target tensor: torch.Size([120]) torch.int64
test input tensor:   torch.Size([30, 4]) torch.float32
test target tensor:  torch.Size([30]) torch.int64
